# Demo en vivo — Fase 2: Predicción de riesgo de ENT (Cesar)

**Asignatura:** Inteligencia Artificial — Universidad Popular del Cesar
**Para:** sustentación oral de la Fase 2 (demo del modelo funcionando, aunque sea parcial)

Este notebook es independiente de `modelado_predictivo_ent_nbi.ipynb`: toma el dataset ya
integrado (`dataset_final_cesar_ent.csv`), reentrena los **dos modelos finales** (los que
mejor resultado dieron en la Fase 2) y deja una **demo interactiva** para correr en vivo
frente al profesor.

- **Regresión** (tasa de atenciones ENT por 1.000 hab.): **Ridge**, alpha = 100.
- **Clasificación** (riesgo Bajo / Medio / Alto): **Gradient Boosting Classifier**,
  n = 100, learning_rate = 0.01, profundidad = 3.

Los dos se entrenan **solo con los 17 municipios de entrenamiento**. La demo predice sobre
los **4 municipios de prueba** (González, Bosconia, Pelaya, La Gloria) — municipios que el
modelo nunca vio al entrenar, para que la predicción sea honesta y no un truco de memoria.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingClassifier

# Resuelve la raiz del proyecto sin importar desde donde se ejecute
def _raiz_proyecto():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "02_datos_procesados").is_dir():
            return p
    return Path.cwd()

BASE = _raiz_proyecto()
DIR_PROC = BASE / "02_datos_procesados"
ruta_dataset = DIR_PROC / "dataset_final_cesar_ent.csv"
if not ruta_dataset.exists():
    ruta_dataset = Path("dataset_final_cesar_ent.csv")  # fallback: mismo directorio

df = pd.read_csv(ruta_dataset, encoding="utf-8-sig")
print("Dataset cargado:", df.shape[0], "municipios,", df.shape[1], "columnas")

Dataset cargado: 25 municipios, 35 columnas


## 1. Mismas variables que en el modelado de la Fase 2

Se reconstruyen las dos brechas rural–urbana y el log de población, y se usa el mismo
conjunto curado de 10 predictores (igual que en `modelado_predictivo_ent_nbi.ipynb`).

In [2]:
df["FE_Brecha_NBI_CPRD_Cabecera"] = df["CPRD_NBI_pct"] - df["Cabecera_NBI_pct"]
df["FE_Brecha_Miseria_CPRD_Cabecera"] = df["CPRD_Miseria_pct"] - df["Cabecera_Miseria_pct"]
df["FE_log_Poblacion"] = np.log1p(df["Poblacion"])

predictores = [
    "Total_NBI_pct", "Total_Miseria_pct", "Total_Vivienda", "Total_Servicios",
    "Total_Hacinamiento", "Total_Inasistencia", "Total_Dependencia",
    "FE_Brecha_NBI_CPRD_Cabecera", "FE_Brecha_Miseria_CPRD_Cabecera", "FE_log_Poblacion",
]

X = df[predictores].copy()
y_reg = np.log(df["Tasa_Total_ENT_1000"])              # el modelo entrena en escala log
y_clf = pd.qcut(df["Tasa_Total_ENT_1000"], q=3, labels=["Bajo", "Medio", "Alto"])
df["Categoria_Riesgo"] = y_clf

# Misma particion que en la Fase 2 (random_state=42): 70% train / 15% val / 15% test
X_train, X_temp, y_train, y_temp, yc_train, yc_temp = train_test_split(
    X, y_reg, y_clf, test_size=0.30, stratify=y_clf, random_state=42)
X_val, X_test, y_val, y_test, yc_val, yc_test = train_test_split(
    X_temp, y_temp, yc_temp, test_size=0.50, stratify=yc_temp, random_state=42)

municipios_train = set(df.loc[X_train.index, "Nom_Mun"])
municipios_test = list(df.loc[X_test.index, "Nom_Mun"])
print("Entrenamiento:", len(X_train), "municipios")
print("Prueba (nunca vistos):", municipios_test)

Entrenamiento: 17 municipios
Prueba (nunca vistos): ['GONZÁLEZ', 'BOSCONIA', 'PELAYA', 'LA GLORIA']


## 2. Entrenar los modelos finales (solo con los datos de entrenamiento)

In [3]:
pipe_ridge = Pipeline([
    ("prep", ColumnTransformer([("num", StandardScaler(), predictores)])),
    ("reg", Ridge(alpha=100)),
])
pipe_ridge.fit(X_train, y_train)

pipe_gbc = Pipeline([
    ("prep", ColumnTransformer([("num", StandardScaler(), predictores)])),
    ("clf", GradientBoostingClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, random_state=42)),
])
pipe_gbc.fit(X_train, yc_train)

print("Modelos entrenados: Ridge (regresion) y Gradient Boosting (clasificacion)")

Modelos entrenados: Ridge (regresion) y Gradient Boosting (clasificacion)


## 3. Función de predicción

Recibe el nombre de un municipio del Cesar (cualquiera de los 25) y muestra:
- la tasa de atenciones ENT predicha (deshaciendo la transformación logarítmica),
- la categoría de riesgo predicha con su probabilidad,
- y, para comparar, el valor real observado.

Si el municipio elegido es uno de los 17 de entrenamiento, se avisa explícitamente —
el modelo ya lo vio, así que ahí la predicción suele ser más optimista de lo normal.

In [4]:
def predecir_municipio(nombre):
    nombre = nombre.strip().upper()
    fila = df[df["Nom_Mun"] == nombre]
    if fila.empty:
        print(f"No encontre '{nombre}'. Municipios disponibles:")
        print(", ".join(sorted(df['Nom_Mun'])))
        return None

    idx = fila.index[0]
    x_fila = df.loc[[idx], predictores]

    tasa_pred = float(np.exp(pipe_ridge.predict(x_fila))[0])
    tasa_real = float(df.loc[idx, "Tasa_Total_ENT_1000"])

    riesgo_pred = pipe_gbc.predict(x_fila)[0]
    proba = pipe_gbc.predict_proba(x_fila)[0]
    clases = pipe_gbc.named_steps["clf"].classes_
    riesgo_real = df.loc[idx, "Categoria_Riesgo"]

    es_entrenamiento = nombre in municipios_train
    print("=" * 56)
    print(f"Municipio: {nombre}", "(usado en ENTRENAMIENTO)" if es_entrenamiento else "(dato de PRUEBA, nunca visto)")
    print("=" * 56)
    print(f"Tasa de atenciones ENT por 1.000 hab.")
    print(f"   Predicha (Ridge):  {tasa_pred:,.1f}")
    print(f"   Observada (real):  {tasa_real:,.1f}")
    print()
    print(f"Categoria de riesgo")
    print(f"   Predicha (Gradient Boosting): {riesgo_pred}")
    print(f"   Observada (real):             {riesgo_real}")
    print(f"   Probabilidades: " + ", ".join(f"{c}={p:.0%}" for c, p in zip(clases, proba)))

    return {"municipio": nombre, "tasa_predicha": tasa_pred, "tasa_real": tasa_real,
            "riesgo_predicho": riesgo_pred, "riesgo_real": riesgo_real,
            "proba": dict(zip(clases, proba)), "entrenamiento": es_entrenamiento}

_ = predecir_municipio("GONZÁLEZ")

Municipio: GONZÁLEZ (dato de PRUEBA, nunca visto)
Tasa de atenciones ENT por 1.000 hab.
   Predicha (Ridge):  1,492.4
   Observada (real):  2,714.2

Categoria de riesgo
   Predicha (Gradient Boosting): Alto
   Observada (real):             Alto
   Probabilidades: Alto=73%, Bajo=16%, Medio=11%


## 4. Resumen de los 4 municipios de prueba

Esta es la tabla que respalda la sección 10.2.5 del documento — generada aquí en vivo,
no copiada.

In [5]:
resumen = []
for m in municipios_test:
    r = predecir_municipio(m)
    print()
    resumen.append(r)

tabla = pd.DataFrame(resumen)[["municipio", "tasa_real", "tasa_predicha", "riesgo_real", "riesgo_predicho"]]
tabla.columns = ["Municipio", "Tasa observada", "Tasa predicha (Ridge)", "Riesgo real", "Riesgo predicho (GB)"]
tabla.round(1)

Municipio: GONZÁLEZ (dato de PRUEBA, nunca visto)
Tasa de atenciones ENT por 1.000 hab.
   Predicha (Ridge):  1,492.4
   Observada (real):  2,714.2

Categoria de riesgo
   Predicha (Gradient Boosting): Alto
   Observada (real):             Alto
   Probabilidades: Alto=73%, Bajo=16%, Medio=11%

Municipio: BOSCONIA (dato de PRUEBA, nunca visto)
Tasa de atenciones ENT por 1.000 hab.
   Predicha (Ridge):  1,213.9
   Observada (real):  1,834.4

Categoria de riesgo
   Predicha (Gradient Boosting): Bajo
   Observada (real):             Alto
   Probabilidades: Alto=9%, Bajo=80%, Medio=11%

Municipio: PELAYA (dato de PRUEBA, nunca visto)
Tasa de atenciones ENT por 1.000 hab.
   Predicha (Ridge):  1,337.5
   Observada (real):  1,475.2

Categoria de riesgo
   Predicha (Gradient Boosting): Medio
   Observada (real):             Medio
   Probabilidades: Alto=12%, Bajo=20%, Medio=68%

Municipio: LA GLORIA (dato de PRUEBA, nunca visto)
Tasa de atenciones ENT por 1.000 hab.
   Predicha (Ridge):  1,362

,Municipio,Tasa observada,Tasa predicha (Ridge),Riesgo real,Riesgo predicho (GB)
0,GONZÁLEZ,2714.2,1492.4,Alto,Alto
1,BOSCONIA,1834.4,1213.9,Alto,Bajo
2,PELAYA,1475.2,1337.5,Medio,Medio
3,LA GLORIA,847.9,1362.7,Bajo,Medio


## 5. Demo interactiva

Corre esta celda y elige cualquier municipio del Cesar en el menú desplegable — se
recomienda probar primero con uno de los **4 de prueba** (los que el modelo nunca vio) y
después, si da tiempo, con uno de entrenamiento para que se note la diferencia.

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output

opciones = []
for m in sorted(df["Nom_Mun"]):
    etiqueta = f"{m} (prueba)" if m in municipios_test else f"{m} (entrenamiento)"
    opciones.append((etiqueta, m))

dropdown = widgets.Dropdown(options=opciones, description="Municipio:",
                             style={"description_width": "initial"}, layout=widgets.Layout(width="420px"))
boton = widgets.Button(description="Predecir", button_style="primary")
salida = widgets.Output()

COLOR_RIESGO = {"Bajo": "#86b6ef", "Medio": "#3987e5", "Alto": "#184f95"}

def on_click(_):
    with salida:
        clear_output(wait=True)
        r = predecir_municipio(dropdown.value)
        if r is None:
            return
        fig, ax = plt.subplots(figsize=(4.5, 2.2))
        clases = list(r["proba"].keys())
        valores = list(r["proba"].values())
        colores = [COLOR_RIESGO.get(c, "#888888") for c in clases]
        ax.barh(clases, valores, color=colores)
        ax.set_xlim(0, 1)
        ax.set_xlabel("probabilidad")
        ax.set_title(f"Probabilidad de cada nivel de riesgo — {r['municipio']}")
        for i, v in enumerate(valores):
            ax.text(v + 0.02, i, f"{v:.0%}", va="center")
        plt.tight_layout()
        plt.show()

boton.on_click(on_click)
display(widgets.VBox([widgets.HBox([dropdown, boton]), salida]))

## 6. Advertencia honesta para la sustentación

Con solo **25 municipios** (17 para entrenar) las métricas son inestables — esta demo
muestra que el modelo funciona de punta a punta, no que sea una herramienta lista para
producción. El hallazgo más importante de la Fase 2 sigue aplicando aquí: la tasa de
atenciones mide en parte **oferta y acceso a servicios de salud**, no solo enfermedad real,
así que una predicción baja en un municipio disperso puede reflejar menos diagnóstico, no
menos riesgo real.